# Community Alignment Dataset — Load & Filter

Goal: load `facebook/community-alignment-dataset`, inspect schema empirically, then produce a
clean filtered parquet of English, first-turn, explanation-present rows from prompts with ≥ 10
distinct annotators.

In [19]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────
import re
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from datasets import load_dataset, get_dataset_config_names, get_dataset_split_names

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
print('imports ok')

imports ok


In [20]:
# ── Cell 2: Dataset loading ───────────────────────────────────────────────
#
# Step 1: check what configs (subsets) the dataset exposes.
# Some HF datasets have a single default config; others have named configs.
# We must inspect before choosing.

DATASET_NAME = 'facebook/community-alignment-dataset'

try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f'Configs available: {configs}')
except Exception as e:
    configs = [None]  # single default config
    print(f'No named configs (single default). ({e})')

# Step 2: inspect splits for the default config.
cfg = configs[0]  # start with first/only config
try:
    splits = get_dataset_split_names(DATASET_NAME, config_name=cfg)
    print(f'Splits in config={cfg!r}: {splits}')
except Exception as e:
    splits = ['train']
    print(f'Could not enumerate splits, defaulting to ["train"]. ({e})')

Configs available: ['default']
Splits in config='default': ['train']


In [21]:
# ── Cell 2b: Load all splits, concatenate ────────────────────────────────
#
# Rationale for combining: we want the maximum annotator diversity for the
# ≥10-annotator filter. Train-only might drop prompts that appear in test.
# We add a 'split' column so we can always filter back if needed.

frames = {}
for sp in splits:
    kwargs = dict(split=sp)
    if cfg is not None:
        kwargs['name'] = cfg
    ds = load_dataset(DATASET_NAME, **kwargs)
    frames[sp] = ds.to_pandas()
    frames[sp].insert(0, 'hf_split', sp)
    print(f'  split={sp!r}: {len(frames[sp]):,} rows')

raw = pd.concat(frames.values(), ignore_index=True)
print(f'\nTotal after concatenating splits: {len(raw):,} rows')

  split='train': 90,256 rows

Total after concatenating splits: 90,256 rows


In [22]:
# ── Cell 3: Schema inspection ─────────────────────────────────────────────

print('=== dtypes ===')
print(raw.dtypes.to_string())

print('\n=== shape ===')
print(raw.shape)

print('\n=== null counts ===')
print(raw.isnull().sum().to_string())

print('\n=== first 3 rows (transposed for readability) ===')
raw.head(3).T

=== dtypes ===
hf_split                          object
conversation_id                    int64
annotator_id                       int64
wave                               int64
assigned_lang                     object
is_pregenerated_first_prompt        bool
annotator_age                     object
annotator_gender                  object
annotator_education_level         object
annotator_political               object
annotator_ethnicity               object
annotator_country                 object
first_turn_prompt                 object
first_turn_responses              object
first_turn_response_a             object
first_turn_response_b             object
first_turn_response_c             object
first_turn_response_d             object
first_turn_preferred_response     object
first_turn_feedback               object
second_turn_prompt                object
second_turn_responses             object
second_turn_response_a            object
second_turn_response_b            object
s

,0,1,2
hf_split,train,train,train
conversation_id,1061830552573006,1740200396922271,663596853320456
annotator_id,61575510183566,61574885769787,61575115079212
wave,1,1,1
assigned_lang,en,fr,it
is_pregenerated_first_prompt,False,True,True
annotator_age,18-34,18-34,18-34
annotator_gender,male,female,male
annotator_education_level,Some or complete graduate degree,Post-secondary graduate,Post-secondary graduate
annotator_political,Prefer not to say,Somewhat left-leaning,I don't think of myself in this way


In [23]:
# ── Cell 4: Candidate-column discovery ───────────────────────────────────
#
# We are looking for four semantic roles:
#   A) language / locale
#   B) turn number or first-turn indicator
#   C) prompt / question ID
#   D) annotator ID
#   E) explanation / rationale text
#
# Strategy: for every column, report (nunique, dtype, 10-sample values).
# Columns with low cardinality are candidates for A/B; high-cardinality
# integer or string IDs are candidates for C/D.

print('=== Per-column cardinality + sample values ===')
for col in raw.columns:
    n_unique = raw[col].nunique(dropna=False)
    sample = raw[col].dropna().unique()[:6].tolist()
    # truncate long strings
    sample_str = [str(v)[:60] for v in sample]
    print(f'{col!r:40s}  nunique={n_unique:>7,}  dtype={str(raw[col].dtype):12s}  ex={sample_str}')

=== Per-column cardinality + sample values ===
'hf_split'                                nunique=      1  dtype=object        ex=['train']
'conversation_id'                         nunique= 90,256  dtype=int64         ex=['1061830552573006', '1740200396922271', '663596853320456', '1309050483489493', '658845496931332', '3930761880475244']
'annotator_id'                            nunique=  3,603  dtype=int64         ex=['61575510183566', '61574885769787', '61575115079212', '61574934126943', '61575291848717', '61574861412012']
'wave'                                    nunique=      2  dtype=int64         ex=['1', '2']
'assigned_lang'                           nunique=      5  dtype=object        ex=['en', 'fr', 'it', 'pt', 'hi']
'is_pregenerated_first_prompt'            nunique=      2  dtype=bool          ex=['False', 'True']
'annotator_age'                           nunique=      5  dtype=object        ex=['18-34', '35-45', '46-54', '55+']
'annotator_gender'                        nuni

In [24]:
# ── Cell 4b: Value-count drill-down for low-cardinality columns ───────────
#
# Focus on columns with ≤ 200 unique values — most likely to be categorical
# columns (language, preferred-response labels, boolean flags, etc.).

LOW_CARD_THRESH = 200

for col in raw.columns:
    if col == 'hf_split':
        continue
    if raw[col].nunique(dropna=False) <= LOW_CARD_THRESH:
        print(f'\n--- {col!r} (nunique={raw[col].nunique()}) ---')
        vc = raw[col].value_counts(dropna=False).head(30)
        print(vc.to_string())

# ── Wide-format turn column inventory ─────────────────────────────────────
# Turns are encoded as column-name prefixes, NOT as row values.
# Print every turn prefix and all its columns + one sample value so we can
# confirm the naming convention before committing to any column choice.
import re as _re

turn_prefixes = sorted(set(
    m.group(0)
    for col in raw.columns
    for m in [_re.match(r'^(\w+?turn_)', col)] if m
))
print(f'\nTurn prefixes found: {turn_prefixes}')
for pfx in turn_prefixes:
    cols = [c for c in raw.columns if c.startswith(pfx)]
    print(f'\n  {pfx!r}  ({len(cols)} columns):')
    for c in cols:
        sample = str(raw[c].dropna().iloc[0])[:100] if raw[c].notna().any() else 'ALL NULL'
        null_pct = raw[c].isna().mean() * 100
        print(f'    {c:60s}  null={null_pct:4.1f}%  ex: {sample}')


--- 'wave' (nunique=2) ---
1    70861
2    19395

--- 'assigned_lang' (nunique=5) ---
en    30856
pt    16123
it    14705
fr    14394
hi    14178

--- 'is_pregenerated_first_prompt' (nunique=2) ---
True     62295
False    27961

--- 'annotator_age' (nunique=4) ---
18-34    44395
35-45    21017
46-54    11100
55+      10126
None      3618

--- 'annotator_gender' (nunique=3) ---
male      51147
female    36935
None       1881
other       293

--- 'annotator_education_level' (nunique=5) ---
Post-secondary graduate             34227
Some or complete graduate degree    29487
(At most) Complete Secondary        17808
Some post-secondary                  5975
None                                 2724
Other                                  35

--- 'annotator_political' (nunique=7) ---
I don't think of myself in this way    21908
Middle-of-the-road, centrist           19580
Somewhat left-leaning                  17431
Somewhat right-leaning                  9737
Prefer not to say              

In [25]:
# ── Cell 4c: Text-like columns — sample 5 values to judge content ─────────
#
# We print a few rows of each string column whose average length > 20 chars.
# This surfaces explanation / rationale columns without guessing.

for col in raw.select_dtypes(include='object').columns:
    if col == 'hf_split':
        continue
    avg_len = raw[col].dropna().astype(str).str.len().mean()
    if avg_len > 20:
        print(f'\n=== {col!r}  (mean_len={avg_len:.0f}) ===')
        for v in raw[col].dropna().head(3):
            print(f'  | {str(v)[:200]}')


=== 'annotator_education_level'  (mean_len=27) ===
  | Some or complete graduate degree
  | Post-secondary graduate
  | Post-secondary graduate

=== 'annotator_political'  (mean_len=25) ===
  | Prefer not to say
  | Somewhat left-leaning
  | I don't think of myself in this way

=== 'first_turn_prompt'  (mean_len=113) ===
  | are EV cars better or hybrid cars?
  | Je planifie un voyage à la Grande Barrière de corail. Quels sont les meilleurs spots de snorkeling ?
  | Qual è la trama del romanzo "Hunger Games"?

=== 'first_turn_responses'  (mean_len=2885) ===
  | # Response A:
EV cars are the better choice for the environment and the future of transportation. They produce zero tailpipe emissions, reducing greenhouse gas emissions and air pollution in urban are
  | # Response A: 

La Grande Barrière de corail est l'un des écosystèmes les plus diversifiés et fascinants de la planète, offrant une expérience de snorkeling inoubliable. Les meilleurs spots de snorkel
  | # Response A: 

La tr

In [26]:
# ── Cell 5: Final column selection ────────────────────────────────────────
#
# This dataset is WIDE-FORMAT: turns are column-name prefixes, not row values.
# Each row = one annotator × one prompt.
#
# Schema (from Cell 4b output):
#   first_turn_response_a/b/c/d       — the four response texts
#   first_turn_preferred_response     — which response this annotator chose
#                                       ('response_a'…'response_d', or NaN if skipped)
#   first_turn_explanation            — free-text reason for the choice (may be absent)
#   second/third/fourth_turn_*        — same pattern for subsequent turns
#
# "First-turn only" here means: rows where the annotator expressed a preference
# on the first turn, i.e. first_turn_preferred_response is not NaN.
#
# EDIT the values below if Cell 4b/4c revealed different column names.

COL_LANGUAGE    = 'assigned_lang'                  # language/locale code
COL_PREFERRED   = 'first_turn_preferred_response'  # 'response_a'…'d' or NaN
COL_EXPLANATION = 'first_turn_feedback'         # free-text rationale (update if needed)
COL_PROMPT_ID   = 'conversation_id'                # prompt/conversation identifier
COL_ANNOTATOR   = 'annotator_id'                   # annotator identifier

ENGLISH_VALUE   = 'en'                             # value in COL_LANGUAGE meaning English

# ── sanity check ──────────────────────────────────────────────────────────
chosen = [COL_LANGUAGE, COL_PREFERRED, COL_EXPLANATION, COL_PROMPT_ID, COL_ANNOTATOR]
missing = [c for c in chosen if c not in raw.columns]
if missing:
    raise ValueError(
        f'Columns not found in dataset: {missing}\n'
        f'Available: {raw.columns.tolist()}'
    )
print('Column selection valid.')
print(f'  language    → {COL_LANGUAGE!r}  (English = {ENGLISH_VALUE!r})')
print(f'  preferred   → {COL_PREFERRED!r}  (non-null = first turn answered)')
print(f'  explanation → {COL_EXPLANATION!r}')
print(f'  prompt_id   → {COL_PROMPT_ID!r}')
print(f'  annotator   → {COL_ANNOTATOR!r}')
print(f'\nPreferred-response value counts (incl. NaN):')
print(raw[COL_PREFERRED].value_counts(dropna=False).to_string())

Column selection valid.
  language    → 'assigned_lang'  (English = 'en')
  preferred   → 'first_turn_preferred_response'  (non-null = first turn answered)
  explanation → 'first_turn_feedback'
  prompt_id   → 'conversation_id'
  annotator   → 'annotator_id'

Preferred-response value counts (incl. NaN):
response_d    31179
response_a    26849
response_b    16047
response_c    15554
None            627


In [27]:
# ── Cell 6: Filtering with row-count diagnostics ──────────────────────────

def report(df, label):
    print(f'{label:55s}  {len(df):>8,} rows')

df = raw.copy()
report(df, 'START (all splits combined)')

# ── 6a: English only ─────────────────────────────────────────────────────
df = df[df[COL_LANGUAGE] == ENGLISH_VALUE]
report(df, f'After {COL_LANGUAGE} == {ENGLISH_VALUE!r}')

# ── 6b: Pre-generated first prompt only ──────────────────────────────────
# is_pregenerated_first_prompt == True means the first-turn prompt was drawn
# from a curated/fixed set rather than entered freely by the annotator.
# Restricting to these gives a controlled prompt distribution.
before_pregen = len(df)
df = df[df['is_pregenerated_first_prompt'] == True]  # noqa: E712
report(df, 'After is_pregenerated_first_prompt == True')
print(f'  (dropped {before_pregen - len(df):,} rows with free-form first prompts)')

# ── 6c: First-turn preference expressed ──────────────────────────────────
# "First turn" is not a row value — it is the PRESENCE of a non-null
# first_turn_preferred_response. Rows where this is NaN are either
# second/third/fourth-turn-only annotations, or skipped first turns.
before_ft = len(df)
df = df[df[COL_PREFERRED].notna()]
report(df, f'After {COL_PREFERRED} is not NaN')
print(f'  (dropped {before_ft - len(df):,} rows with no first-turn preference)')

# sanity: confirm all retained rows have a valid label
assert df[COL_PREFERRED].isin(['response_a', 'response_b', 'response_c', 'response_d']).all(), \
    'Unexpected value in preferred-response column after filter'
print(f'  Preferred-response distribution after filter:')
print(df[COL_PREFERRED].value_counts().to_string())

# ── 6d: Explanation present ───────────────────────────────────────────────
before_expl = len(df)
df = df[df[COL_EXPLANATION].notna() & (df[COL_EXPLANATION].astype(str).str.strip() != '')]
report(df, f'After {COL_EXPLANATION} is non-null/non-empty')
print(f'  (dropped {before_expl - len(df):,} rows with missing explanations)')

START (all splits combined)                                90,256 rows
After assigned_lang == 'en'                                30,856 rows
After is_pregenerated_first_prompt == True                 22,168 rows
  (dropped 8,688 rows with free-form first prompts)
After first_turn_preferred_response is not NaN             22,124 rows
  (dropped 44 rows with no first-turn preference)
  Preferred-response distribution after filter:
response_d    8146
response_a    6945
response_c    3643
response_b    3390
After first_turn_feedback is non-null/non-empty             9,407 rows
  (dropped 12,717 rows with missing explanations)


In [28]:
# ── Cell 7: Prompt-level annotator counting (diagnostic only) ─────────────
#
# No ≥N filter applied — just show the distribution so we understand how
# many annotators each prompt has in the filtered set.

annotators_per_prompt = (
    df.groupby(COL_PROMPT_ID)[COL_ANNOTATOR]
    .nunique()
    .rename('n_annotators')
)

print('Distribution of annotators-per-prompt:')
print(annotators_per_prompt.describe().to_string())
print()

bins = [1, 2, 5, 10, 20, 50, 100, 200, 500, np.inf]
labels = ['1', '2-4', '5-9', '10-19', '20-49', '50-99', '100-199', '200-499', '500+']
binned = pd.cut(annotators_per_prompt, bins=bins, labels=labels, right=False)
print('Annotator-count histogram (# prompts per bin):')
print(binned.value_counts().sort_index().to_string())

df_filtered = df.copy()
print(f'\nNo annotator-count filter applied. Rows retained: {len(df_filtered):,}')

Distribution of annotators-per-prompt:
count    9407.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0

Annotator-count histogram (# prompts per bin):
1          9407
2-4           0
5-9           0
10-19         0
20-49         0
50-99         0
100-199       0
200-499       0
500+          0

No annotator-count filter applied. Rows retained: 9,407


In [29]:
# ── Cell 8: Summary statistics ────────────────────────────────────────────

print('=== Final filtered dataset ===')
print(f'  Rows:        {len(df_filtered):,}')
print(f'  Prompts:     {df_filtered[COL_PROMPT_ID].nunique():,}')
print(f'  Annotators:  {df_filtered[COL_ANNOTATOR].nunique():,}')
print(f'  Splits kept: {df_filtered["hf_split"].value_counts().to_dict()}')
print()

print('=== Explanation length (chars) ===')
df_filtered['expl_len'] = df_filtered[COL_EXPLANATION].astype(str).str.len()
print(df_filtered['expl_len'].describe().round(1).to_string())
print()

print('=== Sample rows (transposed) ===')
df_filtered[[COL_PROMPT_ID, COL_ANNOTATOR, COL_LANGUAGE, COL_EXPLANATION]].head(5).T

=== Final filtered dataset ===
  Rows:        9,407
  Prompts:     9,407
  Annotators:  598
  Splits kept: {'train': 9407}

=== Explanation length (chars) ===
count    9407.0
mean      341.0
std       211.7
min        20.0
25%       207.0
50%       301.0
75%       426.0
max      3497.0

=== Sample rows (transposed) ===


,17,25,56,79,86
conversation_id,702992025732060,695771192796939,654427437500427,1041030501272902,582257200903079
annotator_id,61575131153481,61574767872008,61575257361849,61574928906639,61575197783629
assigned_lang,en,en,en,en,en
first_turn_feedback,Response A resonates most with the initial question by directly addressing h...,"All four responses give good answers to the prompt, but I prefer Response A....","Feels very welcoming and polished, the language felt warm and inspiring with...",I chose my preference as it was the most completed one giving me an actual d...,I prefer Response A because it is comprehensive with a little additional inf...


In [30]:
# ── Cell 9: Save ──────────────────────────────────────────────────────────

import os

OUT_DIR = 'empirics_communityalignment'
os.makedirs(OUT_DIR, exist_ok=True)

PARQUET_PATH = os.path.join(OUT_DIR, 'ca_filtered.parquet')
CSV_PATH     = os.path.join(OUT_DIR, 'ca_sample100.csv')

# drop the helper column before saving
to_save = df_filtered.drop(columns=['expl_len'], errors='ignore')

to_save.to_parquet(PARQUET_PATH, index=False)
print(f'Saved parquet → {PARQUET_PATH}  ({os.path.getsize(PARQUET_PATH)/1e6:.1f} MB)')

# 100-row sample: pick 10 rows from each of 10 random prompts for diversity
sample_prompts = (
    to_save[COL_PROMPT_ID].drop_duplicates()
    .sample(min(10, to_save[COL_PROMPT_ID].nunique()), random_state=42)
)
csv_sample = to_save[to_save[COL_PROMPT_ID].isin(sample_prompts)].head(100)
csv_sample.to_csv(CSV_PATH, index=False)
print(f'Saved CSV sample → {CSV_PATH}  ({len(csv_sample)} rows)')

print('\nDone.')

Saved parquet → empirics_communityalignment/ca_filtered.parquet  (83.7 MB)
Saved CSV sample → empirics_communityalignment/ca_sample100.csv  (10 rows)

Done.
